# How to read a JSON Lines file with Python

Load JSONL into pandas, control the types, stream large files in chunks, filter while reading, and query JSONL with SQL.

This notebook accompanies [How to read a JSON Lines file with Python](https://docs.opteryx.app/docs/guides/read-a-json-lines-file-with-python) on docs.opteryx.app. It builds its own sample data, so it runs top to bottom with nothing to download.

In [ ]:
%pip install -q pandas pyarrow rugo opteryx-core

## Make some sample data

200,000 synthetic orders, sorted by `ordered_at`. Every example below reads this file.

In [ ]:
import datetime
import random

import pyarrow

random.seed(42)
N = 200_000
start = datetime.datetime(2024, 1, 1)

orders = pyarrow.table({
    "order_id": list(range(1, N + 1)),
    "customer_id": [random.randint(1, 20_000) for _ in range(N)],
    "country": [random.choice(["GB", "US", "DE", "FR", "JP", "AU", "CA", "NL"]) for _ in range(N)],
    "status": random.choices(["delivered", "shipped", "pending", "cancelled", "returned"], weights=[60, 15, 10, 10, 5], k=N),
    "amount": [round(random.lognormvariate(3.5, 0.8), 2) for _ in range(N)],
    "ordered_at": sorted(start + datetime.timedelta(seconds=random.randint(0, 365 * 86400)) for _ in range(N)),
})
orders.num_rows

In [ ]:
import json

with open("orders.jsonl", "w") as f:
    for row in orders.to_pylist():
        row["ordered_at"] = row["ordered_at"].isoformat()
        f.write(json.dumps(row) + "\n")

!head -n 2 orders.jsonl

## The short answer: pandas

`lines=True` is the important part. `ordered_at` is converted to a datetime because its name ends in `_at`.

In [ ]:
import pandas as pd

df = pd.read_json("orders.jsonl", lines=True)
df.dtypes

In [ ]:
df.head()

## Control the types

Name date columns with `convert_dates`, and set other types with `dtype`. `nrows` reads just the start of the file.

In [ ]:
df = pd.read_json(
    "orders.jsonl",
    lines=True,
    nrows=1000,
    dtype={"customer_id": "int64", "country": "string"},
)
df.dtypes

## Large files: read in chunks

Memory peaks at one chunk plus the matches. Every line is still parsed before it's filtered.

In [ ]:
parts = []
with pd.read_json("orders.jsonl", lines=True, chunksize=50_000) as reader:
    for chunk in reader:
        parts.append(
            chunk.loc[(chunk["country"] == "GB") & (chunk["amount"] > 100), ["order_id", "amount"]]
        )

df = pd.concat(parts, ignore_index=True)
len(df)

## Standard library only

A generator gives the same streaming filter with no dependencies.

In [ ]:
def matching_orders(path):
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            if row["country"] == "GB" and row["amount"] > 100:
                yield {"order_id": row["order_id"], "amount": row["amount"]}

df = pd.DataFrame(matching_orders("orders.jsonl"))
len(df)

## PyArrow

A native, multi-threaded parse of the whole file, then a filter over Arrow arrays.

In [ ]:
import pyarrow.compute as pc
import pyarrow.json as pj

table = pj.read_json("orders.jsonl")
print(table.schema)

df = table.filter((pc.field("country") == "GB") & (pc.field("amount") > 100)).to_pandas()
len(df)

## Filter while reading: Rugo

Rows that don't match the predicates are never built.

In [ ]:
import pyarrow
from rugo.jsonl import read_jsonl, read_metadata

with read_jsonl(
    "orders.jsonl",
    columns=["order_id", "country", "amount"],
    predicates=[("country", "==", "GB"), ("amount", ">", 100)],
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

len(df)

In [ ]:
# timestamps stay as text unless you declare them
with read_jsonl(
    "orders.jsonl",
    columns=["order_id", "ordered_at"],
    explicit_schema={"ordered_at": "TIMESTAMP[us]"},
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.dtypes

In [ ]:
read_metadata("orders.jsonl")

## With SQL: Opteryx

The `WHERE` clause and the columns used are pushed into the read; only the grouped answer reaches pandas.

In [ ]:
import opteryx

morsels = opteryx.session().execute_to_morsels("""
    SELECT status, COUNT(*) AS orders
      FROM READ_JSONL('orders.jsonl')
     WHERE country = 'GB'
     GROUP BY status
     ORDER BY orders DESC
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()